In [0]:
# Databricks notebook source
from datetime import datetime, timezone
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

RUN_ID = f"uc16_dq_{datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')}"
CHECKED_AT = datetime.now(timezone.utc)

checks = [
    (
        "no_duplicate_appointment_ids",
        "silver.appointments_clean",
        """
        SELECT COUNT(*) AS failed_count
        FROM (
          SELECT appointment_id
          FROM poc_health_no_show.silver.appointments_clean
          GROUP BY appointment_id
          HAVING COUNT(*) > 1
        )
        """
    ),
    (
        "no_negative_age",
        "silver.appointments_clean",
        """
        SELECT COUNT(*) AS failed_count
        FROM poc_health_no_show.silver.appointments_clean
        WHERE age < 0
        """
    ),
    (
        "no_negative_lead_time",
        "gold.appointment_risk_segments",
        """
        SELECT COUNT(*) AS failed_count
        FROM poc_health_no_show.gold.appointment_risk_segments
        WHERE lead_time_days < 0
        """
    ),
    (
        "risk_segment_threshold_mismatch",
        "gold.appointment_risk_segments",
        """
        SELECT COUNT(*) AS failed_count
        FROM poc_health_no_show.gold.appointment_risk_segments
        WHERE CASE
          WHEN risk_score >= 70 THEN risk_segment <> 'HIGH'
          WHEN risk_score >= 50 THEN risk_segment <> 'MEDIUM_HIGH'
          WHEN risk_score >= 30 THEN risk_segment <> 'MEDIUM'
          ELSE risk_segment <> 'LOW'
        END
        """
    ),
    (
        "no_empty_risk_reason",
        "gold.appointment_risk_segments",
        """
        SELECT COUNT(*) AS failed_count
        FROM poc_health_no_show.gold.appointment_risk_segments
        WHERE risk_reason IS NULL OR trim(risk_reason) = ''
        """
    ),
    (
        "no_invalid_risk_segment",
        "gold.appointment_risk_segments",
        """
        SELECT COUNT(*) AS failed_count
        FROM poc_health_no_show.gold.appointment_risk_segments
        WHERE risk_segment NOT IN (
          'HIGH', 'MEDIUM_HIGH', 'MEDIUM', 'LOW'
        )
        """
    )
]

results = []

for check_name, table_name, query in checks:
    failed_count = spark.sql(query).first()["failed_count"]

    total_count = spark.sql(f"""
        SELECT COUNT(*) AS total_count
        FROM poc_health_no_show.{table_name}
    """).first()["total_count"]

    status = "PASS" if failed_count == 0 else "FAIL"

    results.append((
        RUN_ID,
        check_name,
        table_name,
        status,
        failed_count,
        total_count,
        CHECKED_AT
    ))

monitoring_schema = """
  run_id STRING,
  check_name STRING,
  table_name STRING,
  status STRING,
  failed_record_count BIGINT,
  total_record_count BIGINT,
  checked_at TIMESTAMP
"""

monitoring_df = spark.createDataFrame(results, monitoring_schema)

monitoring_df.write.mode("append").saveAsTable(
    "poc_health_no_show.ops.data_quality_results"
)

failed_checks = [
    row for row in results if row[3] == "FAIL"
]

if failed_checks:
    raise Exception(
        f"UC-16 data-quality checks failed: {failed_checks}"
    )

print("All UC-16 data-quality checks passed.")